# Haiku Example 1: CODEX + H&E Patch Visualization

This notebook visualizes preprocessed CODEX and H&E patches from the demo dataset
(region `amm-49489`). The dataset ships with:

- **3 example slices** (`dataset/example_slices/`) for quick inspection
- **Full patch sets** (`dataset/amm-49489/codex_patches/` and `he_patches/`) for whole-region mosaic reconstruction

No raw data staging or rsync is required -- everything is read directly from `Haiku/dataset/`.

In [ ]:
import pickle
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# --------------- paths ---------------
HAIKU_ROOT = Path('/home/yancui/Haiku')
DATASET_ROOT = HAIKU_ROOT / 'dataset'
REGION_ID = 'amm-49489'
EXAMPLE_DIR = HAIKU_ROOT / 'dataset' / 'example_slices'


def norm01(arr):
    """Min-max normalise to [0, 1]."""
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-8)


print(f'Dataset dir : {DATASET_ROOT}')
print(f'Example dir : {EXAMPLE_DIR}')

In [ ]:
# Load the 3 example patches (paired .pkl + .npy)
pkl_files = sorted(EXAMPLE_DIR.glob('*.pkl'))

examples = []
for pf in pkl_files:
    with open(pf, 'rb') as f:
        d = pickle.load(f)
    he = np.load(EXAMPLE_DIR / f'{pf.stem}.npy')
    examples.append({
        'name': pf.stem,
        'codex': d['codex'],
        'markers': d['biomarker_name'],
        'he': he,
    })

print(f'Loaded {len(examples)} example patches')
print(f'Biomarker channels : {examples[0]["codex"].shape[0]}')
print(f'Patch spatial size : {examples[0]["codex"].shape[1]}x{examples[0]["codex"].shape[2]}')
print(f'Markers: {examples[0]["markers"]}')

### Single-Patch Detail: H&E + Biomarker Channels

In [ ]:
row1_markers = ['DAPI', 'CD3e', 'CD8', 'PanCK', 'Ki67', 'CD20']
row2_markers = ['CD4', 'PDL1', 'HLA-DR', 'CollagenIV', 'EpCAM']

for p in examples:
    markers = p['markers']
    codex = p['codex']
    H, W = codex.shape[1], codex.shape[2]

    ncols = 1 + len(row1_markers)  # H&E + 6 markers
    fig, axes = plt.subplots(2, ncols, figsize=(3.2 * ncols, 6.5))

    # --- Row 1: H&E + individual biomarker channels ---
    axes[0, 0].imshow(p['he'])
    axes[0, 0].set_title('H&E', fontsize=10, fontweight='bold', color='#1565c0')
    for spine in axes[0, 0].spines.values():
        spine.set_edgecolor('#1565c0')
        spine.set_linewidth(2)
    axes[0, 0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    for j, m in enumerate(row1_markers):
        mi = markers.index(m)
        axes[0, j + 1].imshow(codex[mi], cmap='magma')
        axes[0, j + 1].set_title(m, fontsize=10, fontweight='bold')
        axes[0, j + 1].axis('off')

    # --- Row 2: RGB composite + remaining channels ---
    ch_r = codex[markers.index('PanCK')].astype(np.float32)
    ch_g = codex[markers.index('CD3e')].astype(np.float32)
    ch_b = codex[markers.index('DAPI')].astype(np.float32)
    comp = np.stack([norm01(ch_r), norm01(ch_g), norm01(ch_b)], axis=-1)
    axes[1, 0].imshow(np.clip(comp, 0, 1))
    axes[1, 0].set_title('R=PanCK G=CD3e B=DAPI', fontsize=8, fontweight='bold', color='#2e7d32')
    for spine in axes[1, 0].spines.values():
        spine.set_edgecolor('#2e7d32')
        spine.set_linewidth(2)
    axes[1, 0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    for j, m in enumerate(row2_markers):
        mi = markers.index(m)
        axes[1, j + 1].imshow(codex[mi], cmap='magma')
        axes[1, j + 1].set_title(m, fontsize=10, fontweight='bold')
        axes[1, j + 1].axis('off')

    # Hide unused axes in row 2
    for j in range(len(row2_markers) + 1, ncols):
        axes[1, j].axis('off')

    fig.suptitle(f'Patch: {p["name"]}  |  {len(markers)} channels',
                 fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

### Whole-Region Mosaic

Below we reconstruct the full tissue view by loading **all** patches for region
`amm-49489` and tiling them according to the spatial coordinates encoded in
each filename (pattern: `region_x1-x2-y1-y2`).

In [ ]:
codex_dir = DATASET_ROOT / 'codex_patches' / REGION_ID
he_dir = DATASET_ROOT / 'he_patches' / REGION_ID

codex_files = sorted(codex_dir.glob('*.pkl'))
he_files = sorted(he_dir.glob('*.npy'))
print(f'CODEX patches: {len(codex_files)}')
print(f'H&E patches  : {len(he_files)}')


def parse_coords(fname):
    """Extract (row_start, col_start) from filename like region_x1-x2-y1-y2."""
    m = re.search(r'_(\d+)-(\d+)-(\d+)-(\d+)', fname)
    if m:
        return int(m.group(1)), int(m.group(3))
    return 0, 0


# Load all patches
patch_data = []
for pf in codex_files:
    r, c = parse_coords(pf.stem)
    with open(pf, 'rb') as f:
        d = pickle.load(f)
    hef = he_dir / f'{pf.stem}.npy'
    he_arr = np.load(hef) if hef.exists() else None
    patch_data.append({
        'r': r, 'c': c,
        'codex': d['codex'],
        'markers': d['biomarker_name'],
        'he': he_arr,
        'name': pf.stem,
    })

# Build spatial grid
rows = sorted(set(p['r'] for p in patch_data))
cols = sorted(set(p['c'] for p in patch_data))
row_idx = {r: i for i, r in enumerate(rows)}
col_idx = {c: i for i, c in enumerate(cols)}
PATCH_SIZE = 256
nr, nc = len(rows), len(cols)

# Key markers for mosaic
key_markers = ['DAPI', 'CD3e', 'CD8', 'PanCK', 'Ki67', 'CD20']
marker_list = patch_data[0]['markers']

# Allocate mosaics
he_mosaic = np.zeros((nr * PATCH_SIZE, nc * PATCH_SIZE, 3), dtype=np.uint8)
marker_mosaics = {m: np.zeros((nr * PATCH_SIZE, nc * PATCH_SIZE), dtype=np.float32)
                  for m in key_markers}

for p in patch_data:
    ri, ci = row_idx[p['r']], col_idx[p['c']]
    r0, r1 = ri * PATCH_SIZE, (ri + 1) * PATCH_SIZE
    c0, c1 = ci * PATCH_SIZE, (ci + 1) * PATCH_SIZE
    if p['he'] is not None:
        he_mosaic[r0:r1, c0:c1] = p['he']
    for m in key_markers:
        if m in p['markers']:
            mi = p['markers'].index(m)
            marker_mosaics[m][r0:r1, c0:c1] = p['codex'][mi]

print(f'Mosaic grid: {nr} x {nc} patches ({nr * PATCH_SIZE} x {nc * PATCH_SIZE} pixels)')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 12))

# --- Row 1: H&E + DAPI + CD3e + CD8 ---
axes[0, 0].imshow(he_mosaic)
axes[0, 0].set_title('H&E (whole region)', fontsize=12, fontweight='bold', color='#1565c0')
for spine in axes[0, 0].spines.values():
    spine.set_edgecolor('#1565c0')
    spine.set_linewidth(2)
axes[0, 0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

for j, m in enumerate(['DAPI', 'CD3e', 'CD8']):
    axes[0, j + 1].imshow(marker_mosaics[m], cmap='magma')
    axes[0, j + 1].set_title(f'{m} (whole region)', fontsize=12, fontweight='bold')
    axes[0, j + 1].axis('off')

# --- Row 2: PanCK + Ki67 + CD20 + RGB composite ---
for j, m in enumerate(['PanCK', 'Ki67', 'CD20']):
    axes[1, j].imshow(marker_mosaics[m], cmap='magma')
    axes[1, j].set_title(f'{m} (whole region)', fontsize=12, fontweight='bold')
    axes[1, j].axis('off')

# RGB composite: R=PanCK, G=CD3e, B=DAPI
rgb = np.stack([norm01(marker_mosaics['PanCK']),
                norm01(marker_mosaics['CD3e']),
                norm01(marker_mosaics['DAPI'])], axis=-1)
axes[1, 3].imshow(np.clip(rgb, 0, 1))
axes[1, 3].set_title('Composite (R=PanCK, G=CD3e, B=DAPI)', fontsize=11,
                      fontweight='bold', color='#2e7d32')
for spine in axes[1, 3].spines.values():
    spine.set_edgecolor('#2e7d32')
    spine.set_linewidth(2)
axes[1, 3].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

fig.suptitle(f'Region: {REGION_ID}  |  {nr}x{nc} patches  |  {len(marker_list)} biomarkers',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()